In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls /content/drive/MyDrive/fast_code_project/

decode_2.cu  decode.cu	proj_conv_silu_kernel.cu


In [34]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
NVIDIA A100-SXM4-80GB


In [35]:
!pip install ninja

## Imports & Global Configuration

In [63]:
import math
import time
import warnings
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

warnings.filterwarnings('ignore')

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.bfloat16          # BF16 matches A100 Tensor Core native format

if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name}")
    print(f"SM count     : {props.multi_processor_count}")
    print(f"VRAM         : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute cap  : {props.major}.{props.minor}")
else:
    print("WARNING: CUDA not available — running on CPU (benchmarks will be slow)")

# ── Architecture constants (Table I) ────────────────────────────────────────
H   = 16      # Number of attention heads
DK  = 128     # Per-head key/query dimension
DV  = 256    # Per-head value dimension
TDK = H * DK  # 2048 — Total key/query projection dim
TDV = H * DV  # 4096 — Total value projection dim
D   = 2048    # Model hidden size

# Benchmark sweep configurations
T_VALS = [1, 2048, 4096, 8192]   # T=1 → decode; larger → prefill
B_VALS = [1, 8, 32, 64]

WARMUP_ITERS = 10
BENCH_ITERS  = 50

print(f"\nArchitecture: H={H}, dk={DK}, dv={DV}, Dk={TDK}, Dv={TDV}, D={D}")

GPU : NVIDIA A100-SXM4-80GB
SM count     : 108
VRAM         : 85.1 GB
Compute cap  : 8.0

Architecture: H=16, dk=128, dv=256, Dk=2048, Dv=4096, D=2048


## Benchmarking Utilities

In [37]:
def benchmark_fn(fn, warmup=WARMUP_ITERS, iters=BENCH_ITERS):
    """
    Time `fn()` with CUDA events for accurate GPU measurement.
    Returns mean latency in milliseconds.
    """
    if DEVICE.type == 'cuda':
        # Warm-up
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()

        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)

        start_ev.record()
        for _ in range(iters):
            fn()
        end_ev.record()
        torch.cuda.synchronize()

        return start_ev.elapsed_time(end_ev) / iters   # ms
    else:
        # CPU fallback
        for _ in range(warmup):
            fn()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        return (time.perf_counter() - t0) / iters * 1e3


def throughput_gb_s(bytes_accessed: int, latency_ms: float) -> float:
    """Convert byte count + latency to memory throughput in GB/s."""
    return bytes_accessed / (latency_ms * 1e-3) / 1e9


def flops_to_tflops(flops: int, latency_ms: float) -> float:
    return flops / (latency_ms * 1e-3) / 1e12


def make_df(results):
    """Convert list-of-dicts to a nicely formatted DataFrame."""
    df = pd.DataFrame(results)
    for col in df.select_dtypes(float).columns:
        df[col] = df[col].map(lambda x: f"{x:.4f}")
    return df


def plot_heatmap(df_raw, value_col, title, fmt=".3f", cmap="viridis"):
    """Pivot B×T results into a heatmap."""
    pivot = df_raw.pivot(index='B', columns='T', values=value_col).astype(float)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(r) for r in pivot.index])
    ax.set_xlabel('Sequence Length T')
    ax.set_ylabel('Batch Size B')
    ax.set_title(title)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            ax.text(j, i, format(v, fmt), ha='center', va='center', fontsize=8,
                    color='white' if v < pivot.values.max() * 0.6 else 'black')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


print("Benchmark utilities defined.")

Benchmark utilities defined.


## Define Customized Kernel

In [90]:
!rm -rf /root/.cache/torch_extensions
!rm -rf /root/.cache/torch_extensions/*

In [94]:
from torch.utils.cpp_extension import load

_KERNEL_PATH = "/content/drive/MyDrive/fast_code_project/decode.cu"

print("Compiling decode.cu …")
_gdn_ext = load(
    name="gdn_decode_ext_0",
    sources=[_KERNEL_PATH],
    extra_cuda_cflags=[
        "-O3",
        "--use_fast_math",
        "-arch=sm_80",  # A100
    ],
    verbose=True
)
print("✓ CUDA extension compiled successfully.")

# python wrapper for customize cuda kernel
def gdn_decode_cuda_kernel(q, k, v, a_scalar, b_scalar, A_log, dt_bias, scale: float, S):
    B  = q.shape[0]
    H  = q.shape[1] # Make sure H is captured
    dv = v.shape[-1]

    o_out = torch.empty(B, H, dv, device=q.device, dtype=torch.float32)
    S_out = torch.empty_like(S)

    _gdn_ext.forward(
        q.contiguous(),
        k.contiguous(),
        v.contiguous(),
        a_scalar.contiguous(),
        b_scalar.contiguous(),
        A_log.contiguous(),
        dt_bias.contiguous(),
        S.contiguous(),
        scale,
        o_out,
        S_out
    )
    return o_out, S_out

Compiling decode.cu …
✓ CUDA extension compiled successfully.


## Define baseline

In [95]:
def gdn_decode_baseline(q, k, v, a_scalar, b_scalar, A_log, dt_bias, scale, S):
    """
    Unfused decode step: each sub-operation is separate.

    q, k  : (B, H, dk)   BF16
    v     : (B, H, dv)   BF16
    a_scalar, b_scalar : (B, H)  BF16 — per-head gate inputs for this token
    A_log, dt_bias     : (H,)    FP32
    scale : scalar
    S     : (B, H, dk, dv) FP32  — in-place modified

    Returns: o (B, H, dv) FP32, updated S
    """
    # Gate computation (FP32)
    raw = a_scalar.float() + dt_bias.float()           # (B, H)
    g   = torch.exp(-torch.exp(A_log.float()) *
                    F.softplus(raw))                    # (B, H)
    beta = torch.sigmoid(b_scalar.float())             # (B, H)

    kf = k.float()   # (B, H, dk)
    vf = v.float()   # (B, H, dv)
    qf = q.float()   # (B, H, dk)

    # Gated decay  (separate kernel in unfused)
    S = S * g[:, :, None, None]

    # Read
    r = torch.einsum('bhk,bhkd->bhd', kf, S)          # (B, H, dv)

    # Delta
    delta = beta[:, :, None] * (vf - r)               # (B, H, dv)

    # Rank-1 update
    S = S + torch.einsum('bhk,bhd->bhkd', kf, delta)

    # Output projection
    o = scale * torch.einsum('bhk,bhkd->bhd', qf, S)  # (B, H, dv)

    return o, S

@torch.compile(fullgraph=True, dynamic=False)
def gdn_decode_fused(q, k, v, a_scalar, b_scalar, A_log, dt_bias, scale, S):
    """
    Fused decode step compiled by torch.compile / inductor.
    All five sub-operations are merged into a minimum number of kernels,
    and the state S is streamed through registers / L2 only once.
    """
    raw  = a_scalar.float() + dt_bias.float()
    g    = torch.exp(-torch.exp(A_log.float()) * F.softplus(raw))
    beta = torch.sigmoid(b_scalar.float())

    kf, vf, qf = k.float(), v.float(), q.float()

    S_decayed = S * g[:, :, None, None]
    r         = torch.einsum('bhk,bhkd->bhd', kf, S_decayed)
    delta     = beta[:, :, None] * (vf - r)
    S_new     = S_decayed + torch.einsum('bhk,bhd->bhkd', kf, delta)
    o         = scale * torch.einsum('bhk,bhkd->bhd', qf, S_new)

    return o, S_new

## Check Correctness

In [96]:
# ── Correctness check ─────────────────────────────────────────────────────────
print("\n── Correctness check ──────────────────────────────────────────────────")
with torch.no_grad():
    _B    = 4
    _q    = torch.randn(_B, H, DK, device=DEVICE, dtype=DTYPE)
    _k    = F.normalize(torch.randn(_B, H, DK, device=DEVICE, dtype=DTYPE), dim=-1)
    _v    = torch.randn(_B, H, DV, device=DEVICE, dtype=DTYPE)
    _a_s  = torch.randn(_B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    _b_s  = torch.randn(_B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    _Alog = torch.randn(H,         device=DEVICE, dtype=torch.float32) * 0.1
    _dtb  = torch.randn(H,         device=DEVICE, dtype=torch.float32) * 0.1
    _S    = torch.zeros(_B, H, DK, DV, device=DEVICE, dtype=torch.float32)
    _scl  = 1.0 / math.sqrt(DK)

    o_base,  S_base  = gdn_decode_baseline(   _q, _k, _v, _a_s, _b_s, _Alog, _dtb, _scl, _S.clone())
    o_fused, S_fused = gdn_decode_fused(      _q, _k, _v, _a_s, _b_s, _Alog, _dtb, _scl, _S.clone())
    o_cuda,  S_cuda  = gdn_decode_cuda_kernel(_q, _k, _v, _a_s, _b_s, _Alog, _dtb, _scl, _S.clone())

    eps = 1e-6
    abs_err_o = (o_base.float() - o_cuda.float()).abs().max().item()
    rel_err_o = ((o_base.float() - o_cuda.float()).abs() /
                 (o_base.float().abs() + eps)).max().item()
    abs_err_S = (S_base.float() - S_cuda.float()).abs().max().item()
    rel_err_S = ((S_base.float() - S_cuda.float()).abs() /
                 (S_base.float().abs() + eps)).max().item()
    o_ok = abs_err_o < 1e-2
    S_ok = abs_err_S < 1e-2

print(
    f"  o: abs={abs_err_o:.2e}  rel={rel_err_o:.2e}  {'✓' if o_ok else '✗ MISMATCH'}\n"
    f"  S: abs={abs_err_S:.2e}  rel={rel_err_S:.2e}  {'✓' if S_ok else '✗ MISMATCH'}"
)


── Correctness check ──────────────────────────────────────────────────
  o: abs=1.19e-07  rel=9.76e-06  ✓
  S: abs=8.94e-08  rel=2.77e-07  ✓


## Benchmark Comparison

In [97]:
# ── Benchmark sweep — only T=1 for decode ────────────────────────────────────

# A100 peak memory bandwidth ≈ 2 TB/s
# A100_BW_TBS = 1.555
A100_BW_GBS = 1953
A100_PEAK_FP32_TFLOPS = 19.5

print("\n── Benchmark sweep ────────────────────────────────────────────────────")
results = []

for B in B_VALS:
    # ── Allocate inputs once per B ────────────────────────────────────────────
    q    = torch.randn(B, H, DK, device=DEVICE, dtype=DTYPE)
    k    = F.normalize(torch.randn(B, H, DK, device=DEVICE, dtype=DTYPE), dim=-1)
    v    = torch.randn(B, H, DV, device=DEVICE, dtype=DTYPE)
    a_s  = torch.randn(B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    b_s  = torch.randn(B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    Alog = torch.randn(H,        device=DEVICE, dtype=torch.float32) * 0.1
    dtb  = torch.randn(H,        device=DEVICE, dtype=torch.float32) * 0.1
    scl  = 1.0 / math.sqrt(DK)

    # S_in is never modified — kernel reads S_in, writes to a new S_out
    # This ensures every benchmark iteration sees the same input state
    S_in = torch.zeros(B, H, DK, DV, device=DEVICE, dtype=torch.float32)

    # Baseline modifies S in-place so we clone each call
    S_base_init = S_in.clone()

    # ── Benchmark functions ───────────────────────────────────────────────────
    def run_base():
        with torch.no_grad():
            gdn_decode_baseline(q, k, v, a_s, b_s, Alog, dtb, scl, S_base_init.clone())

    def run_fused():
        with torch.no_grad():
            gdn_decode_fused(q, k, v, a_s, b_s, Alog, dtb, scl, S_in.clone())

    def run_cuda():
        # S_in is read-only in the kernel — safe to reuse without clone
        gdn_decode_cuda_kernel(q, k, v, a_s, b_s, Alog, dtb, scl, S_in)

    lat_base  = benchmark_fn(run_base)
    lat_fused = benchmark_fn(run_fused)
    lat_cuda  = benchmark_fn(run_cuda)

    # ── Memory traffic ────────────────────────────────────────────────────────
    # S: read once + write once (fp32, 4 bytes)
    state_bytes = B * H * DK * DV * 4 * 2        # read + write
    # q, k: (B, H, DK) bf16 = 2 bytes each
    # v:    (B, H, DV) bf16 = 2 bytes
    # o:    (B, H, DV) fp32 = 4 bytes (output)
    vec_bytes   = B * H * (DK + DK + DV) * 2 + B * H * DV * 4
    total_bytes = state_bytes + vec_bytes

    bw_cuda  = throughput_gb_s(total_bytes, lat_cuda)
    bw_util  = (bw_cuda / A100_BW_GBS) * 100

    # ── FLOPs ─────────────────────────────────────────────────────────────────
    # Per (b,h): gate(1) + retrieve(2) + update(2) + output(2) = 7 × DK × DV
    flops        = B * H * 7 * DK * DV
    tflops_cuda  = flops / (lat_cuda * 1e-3) / 1e12
    util_compute = (tflops_cuda / A100_PEAK_FP32_TFLOPS) * 100

    results.append({
        "B":              B,
        "Baseline (ms)":  round(lat_base,  4),
        "Fused (ms)":     round(lat_fused, 4),
        "CUDA (ms)":      round(lat_cuda,  4),
        "Total Speedup":  round(lat_base  / lat_cuda, 2),
        "Kernel Speedup": round(lat_fused / lat_cuda, 2),
        "BW (GB/s)":      round(bw_cuda,  1),
        "Util %":         round(bw_util,  1),
        "TFLOPs":         round(tflops_cuda, 3),
        "Compute %":      round(util_compute, 1),
    })

    print(
        f"B={B:>2} | "
        f"Base: {lat_base:.4f}ms | "
        f"Fused: {lat_fused:.4f}ms | "
        f"CUDA: {lat_cuda:.4f}ms | "
        f"Speedup: {lat_base/lat_cuda:.1f}x | "
        f"BW: {bw_cuda:.1f}GB/s ({bw_util:.1f}%) | "
        f"TFLOPs: {tflops_cuda:.3f} ({util_compute:.1f}%)"
    )

    del q, k, v, a_s, b_s, Alog, dtb, S_in, S_base_init

# ── Summary table ─────────────────────────────────────────────────────────────
print("\nResults:")
df = pd.DataFrame(results)
print(df.to_string(index=False))


── Benchmark sweep ────────────────────────────────────────────────────
B= 1 | Base: 0.4246ms | Fused: 0.3209ms | CUDA: 0.0171ms | Speedup: 24.8x | BW: 246.6GB/s (12.6%) | TFLOPs: 0.214 (1.1%)
B= 8 | Base: 0.4271ms | Fused: 0.3245ms | CUDA: 0.0324ms | Speedup: 13.2x | BW: 1043.1GB/s (53.4%) | TFLOPs: 0.906 (4.6%)
B=32 | Base: 0.5458ms | Fused: 0.3490ms | CUDA: 0.0975ms | Speedup: 5.6x | BW: 1387.9GB/s (71.1%) | TFLOPs: 1.205 (6.2%)
B=64 | Base: 0.9873ms | Fused: 0.6621ms | CUDA: 0.1809ms | Speedup: 5.5x | BW: 1495.5GB/s (76.6%) | TFLOPs: 1.298 (6.7%)

Results:
 B  Baseline (ms)  Fused (ms)  CUDA (ms)  Total Speedup  Kernel Speedup  BW (GB/s)  Util %  TFLOPs  Compute %
 1         0.4246      0.3209     0.0171          24.77           18.72      246.6    12.6   0.214        1.1
 8         0.4271      0.3245     0.0324          13.17           10.01     1043.1    53.4   0.906        4.6
32         0.5458      0.3490     0.0975           5.60            3.58     1387.9    71.1   1.205    

## Compute Registers

In [43]:
import subprocess

# First find your Python.h path
result = subprocess.run(
    ["find", "/usr", "-name", "Python.h", "-type", "f"],
    capture_output=True, text=True
)
python_include = result.stdout.strip().split('\n')[0]
python_include = '/'.join(python_include.split('/')[:-1])
print(f"Python include: {python_include}")

# Then compile with ptxas verbose
result = subprocess.run([
    "nvcc",
    "-O3",
    "--use_fast_math",
    "-arch=sm_80",
    "--ptxas-options=-v",
    "-x", "cu",
    "--compile",
    "-I/usr/local/lib/python3.12/dist-packages/torch/include",
    "-I/usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include",
    "-I/usr/local/cuda/include",
    f"-I{python_include}",
    "/content/drive/MyDrive/fast_code_project/decode.cu",
    "-o", "/tmp/decode_reg_test.o",
],
capture_output=True, text=True
)

print("STDERR:")
print(result.stderr)

Python include: /usr/local/lib/python3.12/dist-packages/tensorflow/include/external/local_config_python/python_include
STDERR:
ptxas info    : 11 bytes gmem, 88 bytes cmem[4]
ptxas info    : Compiling entry function '_Z17gdn_decode_kernelILi256EEvPK13__nv_bfloat16S2_S2_S2_S2_PKfS4_S4_fPfS5_ii' for 'sm_80'
ptxas info    : Function properties for _Z17gdn_decode_kernelILi256EEvPK13__nv_bfloat16S2_S2_S2_S2_PKfS4_S4_fPfS5_ii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 204 registers, used 1 barriers, 448 bytes cmem[0]
ptxas info    : Compile time = 87.817 ms
ptxas info    : Compiling entry function '_Z17gdn_decode_kernelILi128EEvPK13__nv_bfloat16S2_S2_S2_S2_PKfS4_S4_fPfS5_ii' for 'sm_80'
ptxas info    : Function properties for _Z17gdn_decode_kernelILi128EEvPK13__nv_bfloat16S2_S2_S2_S2_PKfS4_S4_fPfS5_ii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 204 registers, used 1 barriers, 448 bytes cmem[0]
ptxas info